<a href="https://colab.research.google.com/github/Muffalo52/anima_lora-Colab-Trainer/blob/dev/anima_lora_trainer_0628.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# @title 🚀 Anima lora 고효율 학습 파이프라인 (Based on : https://github.com/sorryhyun/anima_lora)

# @markdown ### 1. 🔑 인증 설정 (선택)
HF_TOKEN = "" # @param {type:"string"}
WANDB_API_KEY = "4aa973f834b0ca7186394d01dcabeca9a3c20339" # @param {type:"string"}

# @markdown ### 2. 📁 프로젝트 및 데이터셋 설정
# @markdown `MyDrive/lora_training/datasets/` 경로 내에 있는 폴더명 또는 zip 파일명을 입력하세요.
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "test_dataset" # @param {type:"string"}

# @markdown ### 3. ⚙️ 학습 환경 설정
MIXED_PRECISION = "fp16" # @param ["bf16", "fp16"]
TARGET_COMMIT = "1b2ecc7c7e304bb4b38d861fe78f12401112ecf0" # @param ["HEAD", "1b2ecc7c7e304bb4b38d861fe78f12401112ecf0"]
TORCH_COMPILE = True # @param {type:"boolean"}
COMPILE_INDUCTOR_MODE = "default" # @param ["default", "reduce-overhead", "max-autotune"]
# @markdown `IS_T4_GPU`를 체크하면 코랩 무료/기본 GPU(T4) 환경에 맞춰 FlashAttention을 비활성화하고 호환 모드로 안전하게 구동합니다.
IS_T4_GPU = True # @param {type:"boolean"}
GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}
# @markdown 기존에 학습된 LoRA 파일(`.safetensors`)의 경로를 입력하면, 해당 가중치 위에서부터 새로운 학습을 시작합니다. (새로 학습할 경우 비워두세요)
NETWORK_WEIGHTS = "" # @param {type:"string"}
# # @markdown 기본적으로 제외되는 레이어들을 정규식으로 지정하여 강제로 학습에 포함합니다.
# # @markdown 기본적으로 학습에서 제외되는 시스템 레이어들을 어떻게 처리할지 선택합니다.
INCLUDE_MODE = "\uae30\ubcf8 (\uc81c\uc678 \uc720\uc9c0)"

# @markdown 개별 LoRA 기능을 제어합니다.
USE_ORTHO = False # @param {type:"boolean"}
USE_ORTHO_INIT = False # @param {type:"boolean"}
USE_TIMESTEP_MASK = False # @param {type:"boolean"}
USE_REPA = False # @param {type:"boolean"}
# @markdown ### 🌟 Autoscale (해상도 스케줄링) 설정
AUTOSCALE_MODE = "random" # @param ["none", "curriculum", "random"]
AUTOSCALE_TIERS = "512 1024" # @param {type:"string"}
AUTOSCALE_HIGHRES_RATIO = 0.3 # @param {type:"number"}
AUTOSCALE_RAMP = "step" # @param ["step", "stairs"]

# @markdown ### 4. 🎛️ 기본 하이퍼파라미터 튜닝
TRAIN_BATCH_SIZE = 1 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 4 # @param {type:"integer"}
# @markdown
LEARNING_RATE = "1e-4" # @param {type:"string"}
NETWORK_DIM = 32 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}
# @markdown
MAX_TRAIN_EPOCHS = 80 # @param {type:"integer"}
SAVE_EVERY_N_EPOCHS = 4 # @param {type:"integer"}
CHECKPOINTING_EPOCHS = 9999 # @param {type:"integer"}
# @markdown
OPTIMIZER_TYPE = "AdamW" # @param ["AdamW", "Prodigy", "prodigyplus.ProdigyPlusScheduleFree"]
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear"]
LR_WARMUP_RATIO = 0.1 # @param {type:"slider", min:0.0, max:0.2, step:0.01}
TIMESTEP_SAMPLING = "sigmoid" #@param ["uniform", "sigmoid", "shift", "flux_shift"]
SIGMOID_SCALE = 1.2 #@param {type:"number"}
DISCRETE_FLOW_SHIFT = 1.0 # @param {type:"number"}
# @markdown
BLOCKS_TO_SWAP = 0 # @param {type:"integer"}

# @markdown ### 5. 🧠 Prodigy 옵티마이저 전용 설정
prodigy_d_coef = 2.0 # @param {type:"number"}
prodigy_d0 = 1e-6 # @param {type:"number"}
prodigy_schedulefree_c = 50 # @param {type:"number"}

# @markdown ### 6. 📝 캡션 및 텍스트 증강 설정
CAPTION_DROPOUT_RATE = 0.1 # @param {type:"number"}

import os
import shutil
import subprocess
import re

# 경로 변수 설정
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"

# 1. 드라이브 마운트 및 출력 폴더 생성
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. 환경 변수 설정
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# 3 & 4. 환경 구축 및 의존성 설치
print("\n=== [1~2/5] Environment Setup and Dependencies ===")
!apt install -y pigz aria2 -qq
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv python install 3.13

REPO_URL = "https://github.com/sorryhyun/anima_lora.git"

if not os.path.exists('/content/anima_lora'):
    !git clone {REPO_URL} /content/anima_lora
os.chdir('/content/anima_lora')

if TARGET_COMMIT != "HEAD":
    print(f"체크아웃 진행 중: {TARGET_COMMIT}")
    !git checkout {TARGET_COMMIT}

if os.path.exists('.venv/bin/python'):
    print("Valid virtual environment (.venv) found. Skipping installation.")
else:
    print("Installing environment...")
    !uv venv --python 3.13 --clear
    !uv sync

!uv pip uninstall --python .venv -y rich

# 5. 모델 가중치 다운로드
print("\n=== [3/5] Downloading Pre-trained Models ===")
downloads = [
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors",
        "dest": "models/vae/qwen_image_vae.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors",
        "dest": "models/text_encoders/qwen_3_06b_base.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors",
        "dest": "models/diffusion_models/anima-base-v1.0.safetensors"
    }
]

for item in downloads:
    dest_path = item["dest"]
    url = item["url"]

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    if not os.path.exists(dest_path):
        print(f"⬇️ Downloading: {os.path.basename(dest_path)}")
        aria2_cmd = [
            "aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M",
            "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)
        ]
        if HF_TOKEN.strip():
            aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
        aria2_cmd.append(url)

        try:
            subprocess.run(aria2_cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}. ({e})")
            raise SystemExit("다운로드에 실패하여 프로세스를 종료합니다.")
    else:
        print(f"✅ Already exists: {os.path.basename(dest_path)}")

print("✅ Model download complete.")


# 6. 데이터셋 설정
print("\n=== [4/5] Dataset Preprocessing ===")
if USE_GOOGLE_DRIVE:
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)

    if os.path.exists(DATASET_FOLDER):
        print(f"Copying folder: {DATASET_FOLDER}")
        shutil.copytree(DATASET_FOLDER, LOCAL_DATASET)
    elif os.path.exists(DATASET_ZIP):
        print(f"Extracting archive: {DATASET_ZIP}")
        shutil.unpack_archive(DATASET_ZIP, LOCAL_DATASET)
    else:
        print("Dataset not found. Skipping preprocessing.")

#warmup ratio → warmup steps
supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
images_repeats = {}
if os.path.exists(LOCAL_DATASET):
    # 최상위 폴더 확인
    top_images = [f for f in os.listdir(LOCAL_DATASET) if os.path.isfile(os.path.join(LOCAL_DATASET, f)) and f.lower().endswith(supported_types)]
    if top_images:
        images_repeats[LOCAL_DATASET] = (len(top_images), 1)

    # 하위 폴더 확인
    for item in os.listdir(LOCAL_DATASET):
        item_path = os.path.join(LOCAL_DATASET, item)
        if os.path.isdir(item_path):
            sub_images = [f for f in os.listdir(item_path) if f.lower().endswith(supported_types)]
            if sub_images:
                folder_repeats = 1
                match = re.match(r"^(\d+)_", item)
                if match:
                    folder_repeats = int(match.group(1))
                images_repeats[item_path] = (len(sub_images), folder_repeats)

pre_steps_per_epoch = sum(img * rep for (img, rep) in images_repeats.values())
steps_per_epoch = pre_steps_per_epoch / TRAIN_BATCH_SIZE if pre_steps_per_epoch > 0 else 1
actual_total_steps = int((MAX_TRAIN_EPOCHS * steps_per_epoch) / GRADIENT_ACCUMULATION_STEPS)
lr_warmup_steps = int(actual_total_steps * LR_WARMUP_RATIO)

# ------------------------------------------------------------------------------
# 훈련 파라미터 (train_args) 생성
# ------------------------------------------------------------------------------
optimizer_args_cli = ""
max_grad_norm_cli = ""

if OPTIMIZER_TYPE == "prodigyplus.ProdigyPlusScheduleFree":
    optimizer_args = ["betas=0.95,0.99", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}"]
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    max_grad_norm_cli = "--max_grad_norm 0.0 "

elif OPTIMIZER_TYPE == "Prodigy":
    optimizer_args = [f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}"]

elif OPTIMIZER_TYPE == "AdamW":
    optimizer_args = ["weight_decay=0.1", "betas=0.9,0.99", "fused=True"]


optimizer_args_cli = "--optimizer_args " + " ".join([f'"{arg}"' for arg in optimizer_args]) + " "

compile_args_cli = f"--torch_compile --compile_inductor_mode {COMPILE_INDUCTOR_MODE} " if TORCH_COMPILE else ""
t4_compat_cli = '--attn_mode "torch" ' if IS_T4_GPU else ""
grad_ckpt_cli = "--gradient_checkpointing " if GRADIENT_CHECKPOINTING else ""
unsloth_offload_checkpointing_cli = "--unsloth_offload_checkpointing " if GRADIENT_CHECKPOINTING else ""
save_precision_cli = '--save_precision "fp16" ' if IS_T4_GPU else '--save_precision "bf16" '
network_weights_cli = f'--network_weights "{NETWORK_WEIGHTS.strip()}" ' if NETWORK_WEIGHTS.strip() else ""

network_args_list = []

include_patterns_val = ""
if INCLUDE_MODE == "Modulation 포함":
    include_patterns_val = ".*_modulation.*"
elif INCLUDE_MODE == "전체 포함 (모든 레이어 학습)":
    include_patterns_val = ".*"

if include_patterns_val:
    network_args_list.append(f"include_patterns={include_patterns_val}")

network_args_list.extend([
    f"use_ortho={USE_ORTHO}",
    f"use_ortho_init={USE_ORTHO_INIT}",
    f"use_timestep_mask={USE_TIMESTEP_MASK}",
    f"use_repa={USE_REPA}",
    "min_rank=8",
    "alpha_rank_scale=1.0"
])
if USE_ORTHO or USE_ORTHO_INIT:
    network_args_list.append("down_init=kaiming")
print(f"✅ LoRA 상세 설정 적용: use_ortho={USE_ORTHO}, use_ortho_init={USE_ORTHO_INIT}, use_timestep_mask={USE_TIMESTEP_MASK}, use_repa={USE_REPA}")

network_args_cli = ""
if network_args_list:
    args_joined = " ".join([f"'{arg}'" for arg in network_args_list])
    network_args_cli = f"--network_args {args_joined} "

autoscale_cli = ""
if AUTOSCALE_MODE != "none":
    autoscale_cli = (
        f'--autoscale_mode {AUTOSCALE_MODE} '
        f'--autoscale_highres_ratio {AUTOSCALE_HIGHRES_RATIO} '
        f'--autoscale_ramp {AUTOSCALE_RAMP} '
    )

train_args = (
    f'--output_dir "{OUTPUT_DIR}" '
    f'--output_name "{PROJECT_NAME}" '
    f'--train_batch_size {TRAIN_BATCH_SIZE} '
    f'--network_dim {NETWORK_DIM} '
    f'--network_alpha {NETWORK_ALPHA} '
    f'--learning_rate {LEARNING_RATE} '
    f'--max_train_epochs {MAX_TRAIN_EPOCHS} '
    f'--save_every_n_epochs {SAVE_EVERY_N_EPOCHS} '
    f'--checkpointing_epochs {CHECKPOINTING_EPOCHS} '
    f'--gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} '
    f'{grad_ckpt_cli}'
    f'{unsloth_offload_checkpointing_cli}'
    f'{t4_compat_cli}'
    f'--optimizer_type "{OPTIMIZER_TYPE}" '
    f'{optimizer_args_cli}'
    f'{max_grad_norm_cli}'
    f'{network_args_cli}'
    f'{compile_args_cli}'
    f'--lr_scheduler "{LR_SCHEDULER}" '
    f'--lr_warmup_steps 0 '
    f'--lr_warmup_steps {lr_warmup_steps} '
    f'--timestep_sampling {TIMESTEP_SAMPLING} '
    f'--sigmoid_scale {SIGMOID_SCALE} '
    f'--discrete_flow_shift {DISCRETE_FLOW_SHIFT} '
    f'--blocks_to_swap {BLOCKS_TO_SWAP} '
    f'--mixed_precision "{MIXED_PRECISION}" '
    f'{save_precision_cli}'
    f'{network_weights_cli}'
    f'--caption_dropout_rate {CAPTION_DROPOUT_RATE} '
    f'--console_log_simple '
    f'--validate_every_n_epochs 9999 '
    f'--validation_split_num 0 '
    f'--sample_every_n_epochs 0 '
    f'--sample_every_n_steps 0 '
    f'--no-use_cmmd '
    f'--log_with wandb '
    f'--log_tracker_name "Anima_LoRA_Project" '
    f'--wandb_run_name "{PROJECT_NAME}_Run" '
    f'--wandb_api_key "{WANDB_API_KEY}" '
    f'--log_every_n_steps 1 '
    f'--seed 1557 '
    f'{network_args_cli}'
    f'{autoscale_cli}'
    f'{compile_args_cli}'
)

# ==============================================================================
# 메인 실행 흐름
# ==============================================================================
def start_training_pipeline():
    print("\n🛠️ 내부 패치 적용 중...")

    # os.system("git checkout HEAD library/anima/models.py > /dev/null 2>&1")

    apply_general_optimizations()
    if IS_T4_GPU:
        apply_t4_and_fp16_patches()

    print("\n📦 Downloading Danbooru Tags...")
    !uv run --no-sync python tasks.py download-danbooru-tags

    print("\n🚀 Running image bucketing & VAE latent caching...")
    if AUTOSCALE_MODE != "none":
        print(f"   * Autoscale Tiers: {AUTOSCALE_TIERS}")
        !uv run --no-sync make preprocess-resize ARGS="--autoscale_tiers {AUTOSCALE_TIERS}"
        !uv run --no-sync make preprocess-vae
        !uv run --no-sync make preprocess-te
        !uv run --no-sync make preprocess-pe
    else:
        !uv run --no-sync make preprocess

    if USE_REPA:
        print("\n🚀 Caching REPA Spatial PE features...")
        !uv run --no-sync make preprocess-pe ARGS='--encoder pe_spatial'

    !uv pip install --python .venv wandb prodigy-plus-schedule-free schedulefree

    print("\n=== [5/5] Starting LoRA Model Training ===")
    !uv run --no-sync make lora ARGS="{train_args}"


# ==============================================================================
# 패치 코드 정의부
# ==============================================================================
def apply_general_optimizations():
    print("\n[0/3] 인자 및 로깅 제어 패치 중...")

    # 1. log.py
    log_path = 'library/log.py'
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8") as f: code = f.read()
        if 'raise ImportError("Force disable rich")' in code:
            print("  ⏭️ [스킵] log.py rich 비활성화 (이미 적용됨)")
        elif 'from rich.logging import RichHandler' in code:
            code = code.replace('from rich.logging import RichHandler', 'raise ImportError("Force disable rich")')
            with open(log_path, "w", encoding="utf-8") as f: f.write(code)
            print("  ✅ [성공] log.py rich 비활성화")

    # 2. base.py
    base_py_path = "library/datasets/base.py"
    if os.path.exists(base_py_path):
        with open(base_py_path, "r", encoding="utf-8") as f: code = f.read()
        old_epoch = """                logger.info(
                    "epoch is incremented. current_epoch: {}, epoch: {}".format(
                        self.current_epoch, epoch
                    )
                )
                num_epochs = epoch - self.current_epoch"""
        new_epoch = """                num_epochs = epoch - self.current_epoch"""

        old_warn = """                logger.warning(
                    "epoch is not incremented. current_epoch: {}, epoch: {}".format(
                        self.current_epoch, epoch
                    )
                )
                self.current_epoch = epoch"""
        new_warn = """                self.current_epoch = epoch"""
        if old_epoch in code:
            code = code.replace(old_epoch, new_epoch).replace(old_warn, new_warn)
            with open(base_py_path, "w", encoding="utf-8") as f: f.write(code)
            print("  ✅ [성공] base.py 에포크 로그 삭제")
        else:
            print("  ⏭️ [스킵] base.py 에포크 로그 삭제 (이미 적용됨)")

    # 3. base.toml
    base_toml_path = "configs/base.toml"
    if os.path.exists(base_toml_path):
        with open(base_toml_path, "r", encoding="utf-8") as f: toml_content = f.read()
        new_toml_content = re.sub(r'batch_size\s*=\s*\d+', f'batch_size = {TRAIN_BATCH_SIZE}', toml_content)
        if not TORCH_COMPILE:
            new_toml_content = re.sub(r'torch_compile\s*=\s*true', 'torch_compile = false', new_toml_content)

        if toml_content != new_toml_content:
            with open(base_toml_path, "w", encoding="utf-8") as f: f.write(new_toml_content)
            print("  ✅ [성공] base.toml 제어")
        else:
            print("  ⏭️ [스킵] base.toml 제어 (이미 적용됨)")

    # 4. preprocess.toml
    preprocess_toml_path = "configs/preprocess.toml"
    if os.path.exists(preprocess_toml_path):
        with open(preprocess_toml_path, "r", encoding="utf-8") as f: prep_content = f.read()
        new_prep_content = re.sub(r'min_pixels\s*=\s*[\d\.]+', 'min_pixels = 0', prep_content)
        if prep_content != new_prep_content:
            with open(preprocess_toml_path, "w", encoding="utf-8") as f: f.write(new_prep_content)
            print("  ✅ [성공] preprocess.toml 제어")
        else:
            print("  ⏭️ [스킵] preprocess.toml 제어 (이미 적용됨)")

def apply_t4_and_fp16_patches():
    print("\n🚀 T4 및 FP16 최적화 패치를 시작합니다...\n")

    print("[1/3] flash-attn 제거 중...")
    os.system("uv remove flash-attn > /dev/null 2>&1 || true")
    os.system("uv pip uninstall --python .venv -y flash-attn > /dev/null 2>&1 || true")
    print("  ✅ 완료")

    print("\n[2/3] models.py FP16 NaN 방지 패치 중...")
    models_path = "library/anima/models.py"
    if os.path.exists(models_path):
        with open(models_path, "r", encoding="utf-8") as f: code = f.read()

        # 1. RMSNorm 텐서 캐스팅
        old_rmsnorm = """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float())
        return (output * self.weight).to(x.dtype)"""
        new_rmsnorm = """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32):
            output = self._norm(x.float()).type_as(x)
            return output * self.weight"""
        if old_rmsnorm in code:
            code = code.replace(old_rmsnorm, new_rmsnorm)
            print("  ✅ [성공] RMSNorm 텐서 캐스팅")
        else:
            print("  ⏭️ [스킵] RMSNorm 텐서 캐스팅 (이미 적용됨)")

        # 2. FinalLayer 서명 및 FP32 적용
        old_final_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ):"""
        new_final_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ):"""
        old_final_body = """        if self.use_adaln_lora:
            assert adaln_lora_B_T_3D is not None
            shift_B_T_D, scale_B_T_D = (
                self.adaln_modulation(emb_B_T_D)
                + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
            ).chunk(2, dim=-1)
        else:
            shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)"""
        new_final_body = """        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                assert adaln_lora_B_T_3D is not None
                shift_B_T_D, scale_B_T_D = (
                    self.adaln_modulation(emb_B_T_D)
                    + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
                ).chunk(2, dim=-1)
            else:
                shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)"""
        if old_final_sig in code:
            code = code.replace(old_final_sig, new_final_sig).replace(old_final_body, new_final_body)
            print("  ✅ [성공] FinalLayer 서명 및 FP32 적용")
        else:
            print("  ⏭️ [스킵] FinalLayer 서명 및 FP32 적용 (이미 적용됨)")

        # 3. Block._forward AdaLN 청크 수정
        old_block_sig = """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:"""
        new_block_sig = """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:"""
        old_block_body = """        if self.use_adaln_lora:
            fused_down = self.adaln_fused_down(emb_B_T_D)
            down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
        else:
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
            )"""
        new_block_body = """        if use_fp32:
            x_B_T_H_W_D = x_B_T_H_W_D.float()

        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                fused_down = self.adaln_fused_down(emb_B_T_D)
                down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
            else:
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
                )"""
        if old_block_sig in code:
            code = code.replace(old_block_sig, new_block_sig).replace(old_block_body, new_block_body)
            print("  ✅ [성공] Block._forward AdaLN 청크 수정")
        else:
            print("  ⏭️ [스킵] Block._forward AdaLN 청크 수정 (이미 적용됨)")

        # 4. Block.forward 오프로드 체크포인트
        old_block_fwd_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:"""
        new_block_fwd_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:"""

        old_unsloth = """                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                )"""
        new_unsloth = """                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                )"""

        old_cp1 = """                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )"""
        new_cp1 = """                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )"""

        old_cp2 = """                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )"""
        new_cp2 = """                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )"""

        old_eager = """            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
            )"""
        new_eager = """            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
                use_fp32,
            )"""
        if old_block_fwd_sig in code:
            code = code.replace(old_block_fwd_sig, new_block_fwd_sig)
            code = code.replace(old_unsloth, new_unsloth)
            code = code.replace(old_cp1, new_cp1)
            code = code.replace(old_cp2, new_cp2)
            code = code.replace(old_eager, new_eager)
            print("  ✅ [성공] Block.forward 오프로드 체크포인트 및 서명 변경")
        else:
            print("  ⏭️ [스킵] Block.forward 오프로드 체크포인트 및 서명 변경 (이미 적용됨)")

        # 5. Anima._run_blocks 서명 및 루프
        old_run_sig = """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        **block_kwargs,
    ) -> torch.Tensor:"""
        new_run_sig = """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        use_fp32: bool = False,
        **block_kwargs,
    ) -> torch.Tensor:"""
        old_run_loop = """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                **block_kwargs,
            )"""
        new_run_loop = """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                use_fp32=use_fp32,
                **block_kwargs,
            )"""
        if old_run_sig in code:
            code = code.replace(old_run_sig, new_run_sig).replace(old_run_loop, new_run_loop)
            print("  ✅ [성공] Anima._run_blocks 서명 및 루프 인자 추가")
        else:
            print("  ⏭️ [스킵] Anima._run_blocks 서명 및 루프 인자 추가 (이미 적용됨)")

        # 6. Anima.forward_mini_train_dit 메인 로직
        old_mini_train = r"""(x_B_T_H_W_D = self\._run_blocks\(\s*\n\s*x_B_T_H_W_D,\s*\n\s*t_embedding_B_T_D,\s*\n\s*crossattn_emb,\s*\n\s*attn_params,\s*\n\s*capture_blocks=return_block_features,\s*\n\s*feature_sink=feature_sink,\s*\n\s*stop_after_block=stop_after_block,)"""
        new_mini_train = r"use_fp32 = x_B_T_H_W_D.dtype == torch.float16\n\n        \1\n            use_fp32=use_fp32,"
        if "use_fp32 = x_B_T_H_W_D.dtype == torch.float16" not in code:
            code = re.sub(old_mini_train, new_mini_train, code)
            code = re.sub(
                r"(x_B_T_H_W_O = self\.final_layer\(\s*\n\s*x_B_T_H_W_D,\s*t_emb_final,\s*adaln_lora_B_T_3D=adaln_lora_B_T_3D\s*\n\s*\))",
                r"x_B_T_H_W_O = self.final_layer(\n            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D, use_fp32=use_fp32\n        )",
                code
            )
            print("  ✅ [성공] Anima.forward_mini_train_dit 메인 로직")
        else:
            print("  ⏭️ [스킵] Anima.forward_mini_train_dit 메인 로직 (이미 적용됨)")

        # 7. _make_dynamic_seq_forward 서명 및 전달 수정
        old_dyn_sig = """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
    ):"""
        new_dyn_sig = """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
        use_fp32: bool = False,
    ):"""
        old_dyn_body = """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
        )"""
        new_dyn_body = """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
            use_fp32=use_fp32,
        )"""
        if old_dyn_sig in code:
            code = code.replace(old_dyn_sig, new_dyn_sig).replace(old_dyn_body, new_dyn_body)
            print("  ✅ [성공] _make_dynamic_seq_forward 서명 및 내부 전달 수정")
        else:
            print("  ⏭️ [스킵] _make_dynamic_seq_forward 서명 및 내부 전달 수정 (이미 적용되거나 없음)")

        # 파일 저장
        with open(models_path, "w", encoding="utf-8") as f:
            f.write(code)

    print("\n[3/3] _common.py 정밀도 설정 패치 중...")
    common_path = "scripts/tasks/_common.py"
    if os.path.exists(common_path):
        with open(common_path, "r", encoding="utf-8") as f: code = f.read()

        new_code = re.sub(
            r'(mixed_precision\s*=\s*)["\']bf16["\']',
            rf'\g<1>"{MIXED_PRECISION}"',
            code
        )
        if new_code != code:
            with open(common_path, "w", encoding="utf-8") as f: f.write(new_code)
            print("  ✅ 적용됨: _common.py")
        else:
            if f'"{MIXED_PRECISION}"' in code:
                print("  ⏭️ 스킵 (이미 적용됨): _common.py")
            else:
                print("  ❌ 실패: _common.py에서 'mixed_precision = \"bf16\"' 설정을 찾을 수 없음")

    print("\n🚀 패치 스크립트 실행 종료.")

start_training_pipeline()


=== [1~2/5] Environment Setup and Dependencies ===
aria2 is already the newest version (1.36.0-1).
pigz is already the newest version (2.6-1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
downloading uv 0.11.25 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Python 3.13 is already installed
체크아웃 진행 중: 1b2ecc7c7e304bb4b38d861fe78f12401112ecf0
M	configs/preprocess.toml
M	library/anima/models.py
M	library/datasets/base.py
M	library/log.py
M	pyproject.toml
M	scripts/tasks/_common.py
M	uv.lock
HEAD is now at 1b2ecc7 autoscale?
Valid virtual environment (.venv) found. Skipping installation.

=== [3/5] Downloading Pre-trained Models ===
✅ Already exists: qwen_image_vae.safetensors
✅ Already exists: qwen_3_06b_base.safetensors
✅ Already exists: anima-base-v1.0.safetensors
✅ Model download complete.

=== [4/5] Dataset Preprocessing ===
Extracting archive: /content/drive/MyDrive/lora_training/datasets/test_dataset.zip
✅ LoRA 상세 설정 적용: 